# Profiling visualization notebook
This is just convenience notebook to visualize csv file generated by tracy

Note: This jupyter expect profiler markers inserted by nano_gpt training loop (e.g.: `forward_pass_done`, `backward_pass_done`, etc.)

To generate this csv file, run:
- c++ training: `./tt-train/run_profiler.sh ./build/tt-train/sources/examples/nano_gpt/nano_gpt --config ./tt-train/configs/training_configs/myconfig.aml`
- python training: `./tt-train/run_profiler.sh ./tt-train/sources/examples/nano_gpt/train_nanogpt.py --config ./tt-train/configs/training_configs/myconfig.aml`

In [ ]:
CSV_FILE = "/data/philei/tt-metal/generated/profiler/reports/2026_04_02_11_46_03/ops_perf_results_2026_04_02_11_46_03.csv"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact

# Preprocessing
- Remove everything before `compilation_finished` marker
- Rename some cols for convenience
- Calculate number of training steps

In [ ]:
# Load the CSV file into a pandas DataFrame
df = pd.read_csv(CSV_FILE)

# Rename the columns for clarity
df = df.rename(columns={
    'OP CODE': 'Operation',
    'HOST DURATION [ns]': 'Host Time',
    'OP TO OP LATENCY [ns]': 'Time Between Ops',
    'DEVICE FW DURATION [ns]': 'Device Time'
})


# Filter out rows before compilation finished
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('compilation_finished', na=False)
matching_indices = df.index[mask]
assert not matching_indices.empty, "No 'compilation_finished' found in ProfilerNoopOperation attributes"
latest_compilation_flag = matching_indices[-1]
df = df.iloc[latest_compilation_flag + 1:]

# Find number of training steps
mask = (df['Operation'] == 'ProfilerNoopOperation') & df['ATTRIBUTES'].str.contains('iteration_', na=False)
matching_indices = df.index[mask]
num_training_steps = 3 # for backward compatibility, will be removed before merge
if not matching_indices.empty:
    filtered_df = df[df.index.isin(matching_indices)]
    num_training_steps = len(filtered_df['ATTRIBUTES'].unique())

df = df[df['Operation'] != 'ProfilerNoopOperation']

all_operations = df['Operation'].unique()

print(f"Number of training steps: {num_training_steps}")

In [ ]:
# example how to manually extract performance data for a specific operation
e = df[df['Operation'] == 'SDPAForwardDeviceOperation']
e = e[['Operation', 'Host Time', 'Time Between Ops', 'Device Time']]
e

In [ ]:
DRAW_PIE_CHART = True
DRAW_HISTOGRAMS = True
DRAW_INTERACTIVE_PLOT = True
DRAW_ANOMALIES = False
DRAW_STEP_SEGMENTS_BREAKDOWN = True

# Pie chart
Visualize:
- Sum/Mean of device time for every device operation
- Sum/Mean of host time for every device operation
- Sum/Mean of time between ops (good estimation for dispatch overhead)

In [ ]:
def draw_diagrams_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']
    topk = 15

    for col in time_columns:
        # ----- top-k + “Others” slice -------------------------------------
        sorted_times = grouped[col].sort_values(ascending=False)
        top_times    = sorted_times.head(topk)
        others_sum   = sorted_times.iloc[topk:].sum()
        if others_sum:
            top_times = pd.concat([top_times, pd.Series({'Others': others_sum})])

        labels = top_times.index.tolist()
        sizes  = top_times.values
        pct    = 100 * sizes / sizes.sum()

        # ----- plot -------------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 8))
        wedges, _ = ax.pie(sizes, startangle=140)   # no autopct ⇒ nothing on pie

        legend_text = [f'{lbl} — {p:.1f}%' for lbl, p in zip(labels, pct)]
        ax.legend(
            wedges,
            legend_text,
            title=f'Operations (share of {aggregation})',
            loc='center left',
            bbox_to_anchor=(1, 0.5)
        )

        ax.set_title(f'Top {topk} Operations by {aggregation} {col}')
        ax.axis('equal')
        plt.show()

        # display(fig)    # ➋ show it once

In [ ]:
DRAW_PIE_CHART and draw_diagrams_with_aggregation('sum')

In [ ]:
DRAW_PIE_CHART and draw_diagrams_with_aggregation('mean')

# Historgrams
Visualize the same data as pie charts, but we will see absolute time instead of relative percentage for device operation

In [ ]:
def name_per_aggregation(aggregation):
    if aggregation == 'sum':
        return 'Total (per training step)'
    elif aggregation == 'mean':
        return 'Average'
    else:
        raise ValueError(f"Unsupported aggregation: {aggregation}")
    
def draw_charts_with_aggregation(aggregation):
    grouped = df.groupby('Operation').agg({
        'Host Time': aggregation,
        'Time Between Ops': aggregation,
        'Device Time': aggregation
    })

    time_columns = ['Device Time', 'Host Time', 'Time Between Ops']

    # Loop through each time column to create horizontal bar charts and pie charts
    for col in time_columns:
        # Extract the total times per operation for the current column
        total_times = grouped[col] / 1_000_000  # Convert from nanoseconds to milliseconds
        if aggregation == 'sum':
            total_times /= num_training_steps  # Normalize by number of training steps if aggregation is 'sum'
        
        # Create a horizontal bar chart
        plt.figure(figsize=(10, 6))
        total_times.sort_values().plot(kind='barh', color='skyblue') 
        plt.title(f'{name_per_aggregation(aggregation)} {col} per Operation')
        plt.xlabel(f'{name_per_aggregation(aggregation)} {col} (ms)')
        plt.ylabel('Operation')
        plt.tight_layout()
        plt.show()

In [ ]:
DRAW_HISTOGRAMS and draw_charts_with_aggregation('sum')

In [ ]:
DRAW_HISTOGRAMS and draw_charts_with_aggregation('mean')

# Interactive plot of each op
You can choose every device operation, and see plot of execution time of every instance of such operation vs op index  

Note: Can be quite expensive cell to run

In [ ]:
if DRAW_INTERACTIVE_PLOT:
    @interact(operation=all_operations)
    def draw_per_operation_stats(operation):
        df_op = df[df['Operation'] == operation]
        if df_op.empty:
            print(f"No data available for operation: {operation}")
            return

        metrics = [
            ('Device Time (ms)',        df_op['Device Time']      / 1_000_000),
            ('Host Time (ms)',          df_op['Host Time']        / 1_000_000),
            ('Time Between Ops (ms)',   df_op['Time Between Ops'] / 1_000_000),
        ]

        for title, series in metrics:
            # --- build figure explicitly so we know which one to close ------------
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(series.index, series.values, marker='o', color='red')
            ax.set_title(f'{operation} - {title}')
            ax.set_xlabel('Index')
            ax.set_ylabel(title)
            ax.grid(True)
            fig.tight_layout()
            plt.show()

# Anomaly detection
Next 2 cells try to find anomalies in device/host/between ops time.

Identify anomalies as points that are more than 3 standard deviations from the mean

Note: can take long time to run

In [ ]:
metrics = ['Host Time', 'Time Between Ops', 'Device Time']

def anomaly_detection_per_operation(operation, metric):
    df_op = df[df['Operation'] == operation]
    if df_op.empty:
        print(f"No data available for operation: {operation}")
        return

    # Calculate the mean and standard deviation for each metric
    
    series = df_op[metric]
    
    mean = series.mean()
    std_dev = series.std()
    
    # Identify anomalies as points that are more than 3 standard deviations from the mean
    anomalies = series[(series < mean - 3 * std_dev) | (series > mean + 3 * std_dev)]
    
    if not anomalies.empty:
        return df_op.loc[anomalies.index]
    return None

In [ ]:
import json

def is_not_nan(value):
    """Check if a value is not NaN."""
    return pd.notna(value) and value != 'NaN' and value != 'nan' and value != ''

if DRAW_ANOMALIES:
    @interact(operation=all_operations, metric_name=metrics)
    def show_anomalies_attributes_per_metric(operation, metric_name):
        anomaly_df = anomaly_detection_per_operation(operation, metric_name)

        if anomaly_df is None:
            print(f"No anomalies detected for {operation} in {metric_name}.")
            return

        for index, row in anomaly_df.iterrows():
            attr = row['ATTRIBUTES']
            core_count = row['CORE COUNT']
            metric_value = row[metric_name]

            # find all columns with prefix `INPUT_`
            input_columns = [col for col in row.index if col.startswith('INPUT_')]
            input_values = {col: row[col] for col in input_columns if is_not_nan(row[col])}

            # improve print of dictionary with json
            input_values = json.dumps(input_values, indent=8)

            if isinstance(attr, str):
                attr = attr.replace(';', ',')
                attr = attr.replace('\'', '"')
                try:
                    attr = json.loads(attr)
                except:
                    pass
                attr = json.dumps(attr, indent=8) if isinstance(attr, dict) else attr

            print(f"Anomaly at index {index}: ")
            print(f"    {metric_name} = {metric_value / 1_000_000} ms")
            print(f"    core count = {core_count}")
            print(f"    inputs = {input_values}")
            print(f"    attributes = {attr}")


# Step segments breakdown
Next cell visualized how much time forward/backward/optimizer/gradient_sync took 

In [ ]:
# ── Training Phase Timing Analysis from Profiler Markers ────────────────
# Uses extract_results.py for parsing, then visualizes.

import sys
from pathlib import Path

if DRAW_STEP_SEGMENTS_BREAKDOWN:
    # Infer tt-train path from notebook location: .../tt-train/tools/profiling/
    _nb_dir = Path.cwd() if "__vsc_ipynb_file__" not in dir() else Path(__vsc_ipynb_file__).parent
    TT_TRAIN = (_nb_dir / ".." / "..").resolve()
    if str(TT_TRAIN / "tools") not in sys.path:
        sys.path.insert(0, str(TT_TRAIN / "tools"))

    from profiling.results_json.extract_results import extract_timings, print_timings, PHASE_COLS

    timings = extract_timings(Path(CSV_FILE))
    assert timings is not None, "No timing data found in CSV"

    # Print per-device tables
    print_timings(timings)

    # ── Visualize (first device) ──
    first_dev_key = next(iter(timings))
    dev_data = timings[first_dev_key]
    iters = dev_data["iterations"]
    avail_phases = [c for c in PHASE_COLS if any(it.get(c, 0) > 0 for it in iters)]
    phase_labels = [c.replace("_ms", "").replace("_", " ").title() for c in avail_phases]

    iters_df = pd.DataFrame(iters)
    first_iter = iters_df["iteration"].min()
    steady = iters_df[iters_df["iteration"] > first_iter]
    avg = steady[avail_phases].mean()

    if not steady.empty:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0']

        ax = axes[0]
        avg.plot(kind='bar', ax=ax, color=colors[:len(avail_phases)])
        ax.set_xticklabels(phase_labels, rotation=30, ha='right')
        ax.set_title(f'{first_dev_key}: Average Phase Duration (excl. iter {first_iter})')
        ax.set_ylabel('Time (ms)')
        for i_bar, v in enumerate(avg):
            ax.text(i_bar, v + avg.max() * 0.01, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

        ax = axes[1]
        for ci, col in enumerate(avail_phases):
            ax.plot(steady['iteration'], steady[col], marker='o',
                    label=phase_labels[ci], color=colors[ci % len(colors)])
        ax.set_title(f'{first_dev_key}: Phase Duration per Iteration (excl. iter {first_iter})')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Time (ms)')
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


# Visualize every op in a single image
Creates a big (very tall) image where you can see every single op histogram on y, and execution time on x

Note: Takes a lot of time to run, especially for models with a lot of ops

You can save plot to svg/png or visualize in jupyter directly (see options at the beginning)

In [ ]:
# ── Op-by-op execution timeline for one iteration ──
# Shows each op in execution order with training phase marker
# boundaries and total device time per segment.

if DRAW_STEP_SEGMENTS_BREAKDOWN:
    TIMELINE_DEVICE_ID = 0       # which device to plot
    TIMELINE_ITERATION = 2       # which iteration (post-compilation)
    X_TICK_INTERVAL_MS = 0.5     # x-axis tick spacing in ms
    MARKERS_PER_CALL = 10        # marker rows per profiler call (must match extract_results.py)
    SAVE_PLOT = False             # True: save to file instead of showing inline
    SAVE_FORMAT = "svg"          # "png" or "svg"
    SAVE_DPI = 200               # DPI for png (ignored for svg)

    import re as _re
    import numpy as _np
    from pathlib import Path
    from profiling.results_json.extract_results import PHASE_MAP

    raw_tl = pd.read_csv(CSV_FILE, low_memory=False)
    raw_tl['DEVICE KERNEL DURATION [ns]'] = pd.to_numeric(
        raw_tl['DEVICE KERNEL DURATION [ns]'], errors='coerce'
    )

    # Strip everything up to compilation_finished
    comp_mask = (raw_tl['OP CODE'] == 'ProfilerNoopOperation') & \
                raw_tl['ATTRIBUTES'].str.contains('compilation_finished', na=False)
    comp_indices = raw_tl.index[comp_mask]
    assert not comp_indices.empty, "No compilation_finished marker found"
    raw_tl = raw_tl.iloc[comp_indices[-1] + 1:].reset_index(drop=True)

    # Tag marker identifiers
    raw_tl['_ident'] = None
    noop = raw_tl['OP CODE'] == 'ProfilerNoopOperation'
    raw_tl.loc[noop, '_ident'] = raw_tl.loc[noop, 'ATTRIBUTES'].str.extract(
        r"'identifier':\s*'([^']+)'", expand=False
    )

    iter_start = f'iteration_{TIMELINE_ITERATION}'
    iter_end = f'iteration_{TIMELINE_ITERATION + 1}'
    s_idx = raw_tl.index[raw_tl['_ident'] == iter_start]
    e_idx = raw_tl.index[raw_tl['_ident'] == iter_end]
    assert not s_idx.empty, f"Iteration {TIMELINE_ITERATION} start marker not found"

    s = s_idx[0]
    e = e_idx[0] if not e_idx.empty else len(raw_tl)

    it_df = raw_tl.iloc[s:e].copy().reset_index(drop=True)
    is_marker = it_df['_ident'].notna()

    # Group consecutive marker rows (MARKERS_PER_CALL per profiler call)
    mrows = it_df[is_marker].reset_index()
    marker_groups = []
    mi = 0
    while mi + MARKERS_PER_CALL <= len(mrows):
        ident = mrows.iloc[mi]['_ident']
        grp = mrows.iloc[mi:mi + MARKERS_PER_CALL]
        if (grp['_ident'] == ident).all():
            marker_groups.append({'ident': ident, 'last_pos': int(grp['index'].iloc[-1])})
            mi += MARKERS_PER_CALL
        else:
            mi += 1

    # Non-marker ops for the target device
    ops = it_df[(it_df['DEVICE ID'] == TIMELINE_DEVICE_ID) & (~is_marker)].copy()
    ops = ops.reset_index().rename(columns={'index': 'iter_pos'}).reset_index(drop=True)
    assert not ops.empty, f"No ops for device {TIMELINE_DEVICE_ID} in iteration {TIMELINE_ITERATION}"

    ops['duration_ms'] = (ops['DEVICE KERNEL DURATION [ns]'] / 1e6).fillna(0)

    # Map each marker to a y-position among the plotted ops
    marker_positions = []
    for mg in marker_groups:
        y = int((ops['iter_pos'] <= mg['last_pos']).sum()) - 0.5
        marker_positions.append({'ident': mg['ident'], 'y': y})

    # Build phase segments between consecutive markers
    segments = []
    for si in range(len(marker_groups) - 1):
        a_pos, b_pos = marker_groups[si]['last_pos'], marker_groups[si + 1]['last_pos']
        seg_ops = ops[(ops['iter_pos'] > a_pos) & (ops['iter_pos'] < b_pos)]

        a_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si]['ident'])
        b_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si + 1]['ident'])
        phase = PHASE_MAP.get((a_norm, b_norm), f"{a_norm} -> {b_norm}")
        label = phase.replace('_ms', '').replace('_', ' ').title() if phase.endswith('_ms') else phase

        segments.append({
            'phase': phase, 'label': label,
            'total_ms': seg_ops['duration_ms'].sum(), 'n_ops': len(seg_ops),
            'y_start': marker_positions[si]['y'], 'y_end': marker_positions[si + 1]['y'],
        })

    # Remaining ops after last marker
    if marker_groups:
        rem = ops[ops['iter_pos'] > marker_groups[-1]['last_pos']]
        if not rem.empty:
            segments.append({
                'phase': 'remaining', 'label': 'Remaining',
                'total_ms': rem['duration_ms'].sum(), 'n_ops': len(rem),
                'y_start': marker_positions[-1]['y'], 'y_end': len(ops) - 0.5,
            })

    # ── Plot ──
    fig_height = max(6, len(ops) * 0.12)
    fig, ax = plt.subplots(figsize=(18, fig_height))

    y_pos = range(len(ops))
    q95 = ops['duration_ms'].quantile(0.95)
    bar_colors = ['#F44336' if d > q95 else '#2196F3' for d in ops['duration_ms']]

    ax.barh(y_pos, ops['duration_ms'], color=bar_colors, height=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ops['OP CODE'], fontsize=6)
    ax.invert_yaxis()

    x_max = ops['duration_ms'].max()
    ax.set_xticks(_np.arange(0, x_max + X_TICK_INTERVAL_MS, X_TICK_INTERVAL_MS))
    ax.set_xlabel('Duration (ms)')
    ax.set_title(
        f'Device {TIMELINE_DEVICE_ID}, Iteration {TIMELINE_ITERATION}: '
        f'Op Execution Timeline ({len(ops)} ops, total={ops["duration_ms"].sum():.1f} ms)'
    )
    ax.grid(True, axis='x', alpha=0.3)

    # Phase background bands
    band_colors = ['#BBDEFB', '#FFE0B2', '#C8E6C9', '#FFCDD2', '#E1BEE7', '#FFF9C4']
    for si, seg in enumerate(segments):
        ax.axhspan(seg['y_start'], seg['y_end'], alpha=0.15,
                color=band_colors[si % len(band_colors)], zorder=0)

    # Marker lines + right-side labels
    for mp in marker_positions:
        ax.axhline(y=mp['y'], color='#D32F2F', linewidth=2, linestyle='--', zorder=5)
        ax.annotate(
            f"  {mp['ident']}",
            xy=(1.01, mp['y']), xycoords=ax.get_yaxis_transform(),
            fontsize=8, fontweight='bold', color='#D32F2F',
            va='center', ha='left', annotation_clip=False,
        )

    # Segment phase labels with device time
    for si, seg in enumerate(segments):
        mid_y = (seg['y_start'] + seg['y_end']) / 2
        ax.annotate(
            f"{seg['label']}\n{seg['total_ms']:.2f} ms  ({seg['n_ops']} ops)",
            xy=(1.14, mid_y), xycoords=ax.get_yaxis_transform(),
            fontsize=7, va='center', ha='left', annotation_clip=False,
            bbox=dict(boxstyle='round,pad=0.3',
                    facecolor=band_colors[si % len(band_colors)], alpha=0.5, edgecolor='none'),
        )

    fig.tight_layout()
    fig.subplots_adjust(right=0.68)

    if SAVE_PLOT:
        out_dir = Path(CSV_FILE).parent
        out_name = f"op_timeline_dev{TIMELINE_DEVICE_ID}_iter{TIMELINE_ITERATION}.{SAVE_FORMAT}"
        out_path = out_dir / out_name
        save_kwargs = {'bbox_inches': 'tight'}
        if SAVE_FORMAT == 'png':
            save_kwargs['dpi'] = SAVE_DPI
        fig.savefig(out_path, **save_kwargs)
        plt.close(fig)
        print(f"Saved {fig_height:.0f}×18 inch plot ({len(ops)} ops) → {out_path}")
    else:
        plt.show()

    # Summary table
    print(f"\n{'Phase':<25s} {'Device Time (ms)':>16s} {'Ops':>6s}")
    print("\u2500" * 50)
    for seg in segments:
        print(f"{seg['label']:<25s} {seg['total_ms']:>16.2f} {seg['n_ops']:>6d}")
    print("\u2500" * 50)
    total_ms = sum(s['total_ms'] for s in segments)
    total_ops = sum(s['n_ops'] for s in segments)
    print(f"{'Total':<25s} {total_ms:>16.2f} {total_ops:>6d}")

    print(f"\nTop 10 slowest ops:")
    top = ops.nlargest(10, 'duration_ms')[['OP CODE', 'duration_ms']].reset_index(drop=True)
    top.index += 1
    display(top)


In [ ]:
# # ── Zone summary: all marker segments with device time and op counts ──
# # Reuses raw_tl, marker detection, and ops from the previous cell.

# SUMMARY_DEVICE_ID = TIMELINE_DEVICE_ID
# SUMMARY_ITERATION = TIMELINE_ITERATION

# import re as _re
# from profiling.results_json.extract_results import PHASE_MAP

# # Build a flat list of all ops (non-marker) with their zone assignment
# zone_records = []
# for si in range(len(marker_groups) - 1):
#     a_pos, b_pos = marker_groups[si]['last_pos'], marker_groups[si + 1]['last_pos']
#     seg_ops = ops[(ops['iter_pos'] > a_pos) & (ops['iter_pos'] < b_pos)].copy()

#     a_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si]['ident'])
#     b_norm = _re.sub(r'iteration_\d+', 'iteration', marker_groups[si + 1]['ident'])
#     phase = PHASE_MAP.get((a_norm, b_norm), f"{marker_groups[si]['ident']} -> {marker_groups[si + 1]['ident']}")

#     seg_ops['zone'] = phase
#     seg_ops['zone_order'] = si
#     zone_records.append(seg_ops)

# # Remaining ops after last marker
# if marker_groups:
#     rem = ops[ops['iter_pos'] > marker_groups[-1]['last_pos']].copy()
#     if not rem.empty:
#         rem['zone'] = 'remaining'
#         rem['zone_order'] = len(marker_groups) - 1
#         zone_records.append(rem)

# zone_df = pd.concat(zone_records, ignore_index=True)

# # ── 1. Per-occurrence zone summary (every segment individually) ──
# zone_summary = (
#     zone_df.groupby(['zone_order', 'zone'])
#     .agg(
#         total_device_ms=('duration_ms', 'sum'),
#         num_ops=('duration_ms', 'count'),
#         mean_op_ms=('duration_ms', 'mean'),
#         max_op_ms=('duration_ms', 'max'),
#     )
#     .reset_index()
#     .sort_values('zone_order')
#     .drop(columns='zone_order')
#     .reset_index(drop=True)
# )

# # ── 2. Aggregated zone summary (repeated zones merged) ──
# zone_agg = (
#     zone_df.groupby('zone')
#     .agg(
#         total_device_ms=('duration_ms', 'sum'),
#         num_ops=('duration_ms', 'count'),
#         mean_op_ms=('duration_ms', 'mean'),
#         max_op_ms=('duration_ms', 'max'),
#     )
#     .reset_index()
#     .sort_values('total_device_ms', ascending=False)
#     .reset_index(drop=True)
# )
# zone_agg['occurrences'] = zone_agg['zone'].map(zone_summary['zone'].value_counts())
# zone_agg['pct'] = 100.0 * zone_agg['total_device_ms'] / zone_agg['total_device_ms'].sum()

# # ── 3. Aggregated per-zone per-op breakdown ──
# zone_op_agg = (
#     zone_df.groupby(['zone', 'OP CODE'])
#     .agg(
#         total_ms=('duration_ms', 'sum'),
#         count=('duration_ms', 'count'),
#         mean_ms=('duration_ms', 'mean'),
#         max_ms=('duration_ms', 'max'),
#     )
#     .reset_index()
#     .sort_values(['zone', 'total_ms'], ascending=[True, False])
#     .reset_index(drop=True)
# )

# # ── 4. Per-occurrence per-zone per-op breakdown ──
# zone_op_summary = (
#     zone_df.groupby(['zone_order', 'zone', 'OP CODE'])
#     .agg(
#         total_ms=('duration_ms', 'sum'),
#         count=('duration_ms', 'count'),
#         mean_ms=('duration_ms', 'mean'),
#         max_ms=('duration_ms', 'max'),
#     )
#     .reset_index()
#     .sort_values(['zone_order', 'total_ms'], ascending=[True, False])
#     .drop(columns='zone_order')
#     .reset_index(drop=True)
# )

# print(f"Device {SUMMARY_DEVICE_ID}, Iteration {SUMMARY_ITERATION}")

# print(f"\n{'='*80}")
# print("Aggregated Zone Summary (repeated zones merged, sorted by total time)")
# print(f"{'='*80}")
# display(zone_agg.style.format({
#     'total_device_ms': '{:.2f}',
#     'mean_op_ms': '{:.4f}',
#     'max_op_ms': '{:.4f}',
#     'pct': '{:.1f}%',
# }).hide(axis='index'))

# print(f"\n{'='*80}")
# print("Aggregated Per-Zone Op Breakdown (repeated zones merged)")
# print(f"{'='*80}")
# display(zone_op_agg.style.format({
#     'total_ms': '{:.4f}',
#     'mean_ms': '{:.4f}',
#     'max_ms': '{:.4f}',
# }).hide(axis='index'))

# print(f"\n{'='*80}")
# print("Per-Occurrence Zone Summary (every segment individually)")
# print(f"{'='*80}")
# display(zone_summary.style.format({
#     'total_device_ms': '{:.2f}',
#     'mean_op_ms': '{:.4f}',
#     'max_op_ms': '{:.4f}',
# }).hide(axis='index'))

# print(f"\n{'='*80}")
# print("Per-Occurrence Op Breakdown (sorted by total device time within each zone)")
# print(f"{'='*80}")
# display(zone_op_summary.style.format({
#     'total_ms': '{:.4f}',
#     'mean_ms': '{:.4f}',
#     'max_ms': '{:.4f}',
# }).hide(axis='index'))


In [ ]:
# ── Top-10 Ops by Dispatch Overhead ──────────────────────────────────────────
# Overhead = Op-to-Op Latency (device-side gap between consecutive kernel executions).
# This is the true device idle time caused by dispatch; host times shown for context.

df_oh = df[['Operation', 'Host Time', 'Time Between Ops', 'Device Time']].copy()
df_oh['Device Time'] = df_oh['Device Time'].fillna(0)
df_oh['Time Between Ops'] = df_oh['Time Between Ops'].fillna(0)

NS_TO_MS = 1e-6
n_iters = num_training_steps

grp = df_oh.groupby('Operation')

stats = pd.DataFrame({
    'Calls/Step':           (grp['Host Time'].count() / n_iters).astype(int),
    'Host Avg (ms)':        (grp['Host Time'].mean()  * NS_TO_MS).round(3),
    'Host Total (ms)':      (grp['Host Time'].sum()   / n_iters * NS_TO_MS).round(2),
    'Op2Op Avg (ms)':       (grp['Time Between Ops'].mean() * NS_TO_MS).round(3),
    'Op2Op Total (ms)':     (grp['Time Between Ops'].sum()  / n_iters * NS_TO_MS).round(2),
    'Dev Avg (ms)':         (grp['Device Time'].mean() * NS_TO_MS).round(3),
    'Dev Total (ms)':       (grp['Device Time'].sum()  / n_iters * NS_TO_MS).round(2),
})
stats['Overhead (ms)'] = stats['Op2Op Total (ms)']
stats['Overhead %'] = (
    stats['Op2Op Total (ms)'] / (stats['Op2Op Total (ms)'] + stats['Dev Total (ms)']) * 100
).round(1)

top10 = stats.nlargest(10, 'Overhead (ms)')

# Whole-step summary row
t_host  = df_oh['Host Time'].sum()         / n_iters
t_op2op = df_oh['Time Between Ops'].sum()  / n_iters
t_dev   = df_oh['Device Time'].sum()       / n_iters

summary = pd.DataFrame([{
    'Calls/Step':       int(len(df_oh) / n_iters),
    'Host Avg (ms)':    round(df_oh['Host Time'].mean()         * NS_TO_MS, 3),
    'Host Total (ms)':  round(t_host  * NS_TO_MS, 2),
    'Op2Op Avg (ms)':   round(df_oh['Time Between Ops'].mean() * NS_TO_MS, 3),
    'Op2Op Total (ms)': round(t_op2op * NS_TO_MS, 2),
    'Dev Avg (ms)':     round(df_oh['Device Time'].mean()       * NS_TO_MS, 3),
    'Dev Total (ms)':   round(t_dev   * NS_TO_MS, 2),
    'Overhead (ms)':    round(t_op2op * NS_TO_MS, 2),
    'Overhead %':       round(t_op2op / (t_op2op + t_dev) * 100, 1),
}], index=['── WHOLE STEP ──'])

result = pd.concat([top10, summary])

with pd.option_context('display.max_columns', None, 'display.width', 200):
    display(result)

In [ ]:
# ── [START]/[END] Zone Profiling ─────────────────────────────────────────────
# Pairs explicit [START]/[END] markers to measure device time per named zone.
# Uses stack-based pairing: each [START] is matched with the immediately
# following [END] of the same name, which correctly handles skipped zones
# (e.g., `continue` before [END]) and nested zones.
# Reuses marker_groups and ops from the timeline cell.

import re as _re

ZONE_DEVICE_ID = TIMELINE_DEVICE_ID
ZONE_ITERATION = TIMELINE_ITERATION

# ── 1. Stack-based pairing of [START]/[END] markers ──
# Markers now have [FWD]/[BWD] prefix from profiler_marker():
#   Forward:  [FWD] [START] zone ... [FWD] [END] zone
#   Backward: [BWD] [END] zone ... [BWD] [START] zone  (reversed!)
# Also supports legacy format without [FWD]/[BWD] prefix.
open_stacks = {}
records = []
unpaired_starts = 0

for mg in marker_groups:
    # Try new format: [FWD|BWD] [START|END] zone
    m = _re.match(r'\[(FWD|BWD)\]\s*\[(START|END)\]\s*(.*)', mg['ident'])
    if m:
        pass_type, tag, zone = m.group(1), m.group(2), m.group(3).strip()
        # In backward pass, the autograd graph is reversed:
        # [BWD] [END] fires first (opens zone), [BWD] [START] fires second (closes zone)
        is_open = (tag == 'START') if pass_type == 'FWD' else (tag == 'END')
        zone_key = f"[{pass_type}] {zone}"
    else:
        # Legacy format: [START|END] zone (no FWD/BWD prefix)
        m = _re.match(r'\[(START|END)\]\s*(.*)', mg['ident'])
        if not m:
            continue
        tag, zone = m.group(1), m.group(2).strip()
        is_open = (tag == 'START')
        zone_key = zone

    if is_open:
        open_stacks.setdefault(zone_key, []).append(mg['last_pos'])
    else:
        stack = open_stacks.get(zone_key, [])
        if stack:
            start_pos = stack.pop()
            seg = ops[(ops['iter_pos'] > start_pos) & (ops['iter_pos'] < mg['last_pos'])]
            records.append({
                'zone': zone_key,
                'device_ms': seg['duration_ms'].sum(),
                'num_ops': len(seg),
            })

# Count unpaired markers
for zone, stack in open_stacks.items():
    if stack:
        unpaired_starts += len(stack)

# ── 2. Add train-loop phases from consecutive markers ──
# Forward: dataloader_step_done → forward_pass_done
# Backward: forward_pass_done → backward_pass_done
# Optimizer: backward_pass_done → optimizer_step_done
TRAIN_PHASES = [
    ('dataloader_step_done', 'forward_pass_done',    'Forward (full)'),
    ('forward_pass_done',    'backward_pass_done',   'Backward (full)'),
    ('backward_pass_done',   'optimizer_step_done',  'Optimizer Step (full)'),
]

marker_pos_by_ident = {}
for mg in marker_groups:
    marker_pos_by_ident.setdefault(mg['ident'], []).append(mg['last_pos'])

for start_id, end_id, label in TRAIN_PHASES:
    starts = marker_pos_by_ident.get(start_id, [])
    ends = marker_pos_by_ident.get(end_id, [])
    for i in range(min(len(starts), len(ends))):
        seg = ops[(ops['iter_pos'] > starts[i]) & (ops['iter_pos'] < ends[i])]
        records.append({
            'zone': label,
            'device_ms': seg['duration_ms'].sum(),
            'num_ops': len(seg),
        })

assert records, "No [START]/[END] marker pairs found in this iteration."
zone_raw = pd.DataFrame(records)

# ── 3. Aggregate repeated zones ──
zone_agg = (
    zone_raw.groupby('zone')
    .agg(
        occurrences=('device_ms', 'count'),
        total_device_ms=('device_ms', 'sum'),
        avg_device_ms=('device_ms', 'mean'),
        min_device_ms=('device_ms', 'min'),
        max_device_ms=('device_ms', 'max'),
        total_ops=('num_ops', 'sum'),
    )
    .reset_index()
)

total_step_ms = ops['duration_ms'].sum()
zone_agg['% of step'] = 100.0 * zone_agg['total_device_ms'] / total_step_ms
zone_agg = zone_agg.sort_values('% of step', ascending=False).reset_index(drop=True)

# ── 4. Display ──
print(f"Device {ZONE_DEVICE_ID}, Iteration {ZONE_ITERATION}")
print(f"Total step device time: {total_step_ms:.2f} ms ({len(ops)} ops)")
print(f"Zones found: {len(zone_agg)}, from {len(records)} [START]/[END] pairs")
if unpaired_starts:
    print(f"  ⚠ {unpaired_starts} unpaired [START] markers (missing [END], e.g., from early `continue`)")
print()

display(zone_agg.style.format({
    'total_device_ms': '{:.2f}',
    'avg_device_ms': '{:.4f}',
    'min_device_ms': '{:.4f}',
    'max_device_ms': '{:.4f}',
    '% of step': '{:.1f}%',
}).hide(axis='index'))

In [ ]:
# # ── Zone Tree Visualization ──────────────────────────────────────────────────
# # Builds a nested tree from [START]/[END] zones + train-loop phases.
# # Parent-child nesting is determined automatically from marker position ranges.
# # Reuses marker_groups, ops, total_step_ms from earlier cells.

# import re as _re
# from collections import defaultdict

# # ── 1. Rebuild zone pairs WITH positions ──
# _zone_pairs = []  # (zone_key, start_pos, end_pos, device_ms, num_ops)

# _stacks = {}
# for mg in marker_groups:
#     m = _re.match(r'\[(FWD|BWD)\]\s*\[(START|END)\]\s*(.*)', mg['ident'])
#     if m:
#         pt, tag, zone = m.group(1), m.group(2), m.group(3).strip()
#         is_open = (tag == 'START') if pt == 'FWD' else (tag == 'END')
#         zk = f"[{pt}] {zone}"
#     else:
#         m = _re.match(r'\[(START|END)\]\s*(.*)', mg['ident'])
#         if not m:
#             continue
#         tag, zone = m.group(1), m.group(2).strip()
#         is_open = (tag == 'START')
#         zk = zone

#     if is_open:
#         _stacks.setdefault(zk, []).append(mg['last_pos'])
#     else:
#         st = _stacks.get(zk, [])
#         if st:
#             sp = st.pop()
#             seg = ops[(ops['iter_pos'] > sp) & (ops['iter_pos'] < mg['last_pos'])]
#             _zone_pairs.append((zk, sp, mg['last_pos'], seg['duration_ms'].sum(), len(seg)))

# # Train-loop phases
# _mpos = {}
# for mg in marker_groups:
#     _mpos.setdefault(mg['ident'], []).append(mg['last_pos'])

# for sid, eid, label in [
#     ('dataloader_step_done', 'forward_pass_done', 'Forward (full)'),
#     ('forward_pass_done', 'backward_pass_done', 'Backward (full)'),
#     ('backward_pass_done', 'optimizer_step_done', 'Optimizer Step (full)'),
# ]:
#     ss, es = _mpos.get(sid, []), _mpos.get(eid, [])
#     for i in range(min(len(ss), len(es))):
#         seg = ops[(ops['iter_pos'] > ss[i]) & (ops['iter_pos'] < es[i])]
#         _zone_pairs.append((label, ss[i], es[i], seg['duration_ms'].sum(), len(seg)))

# # Whole step as root
# _zone_pairs.append(('Whole Step', -1, ops['iter_pos'].max() + 1, total_step_ms, len(ops)))

# # ── 2. Aggregate totals and pick representative positions per zone ──
# ztotals = defaultdict(lambda: {'ms': 0.0, 'ops': 0, 'count': 0})
# zrepr = {}

# for zk, sp, ep, ms, no in _zone_pairs:
#     ztotals[zk]['ms'] += ms
#     ztotals[zk]['ops'] += no
#     ztotals[zk]['count'] += 1
#     if zk not in zrepr:
#         zrepr[zk] = (sp, ep)

# # ── 3. Build parent→children tree via position containment ──
# znames = list(ztotals.keys())
# parents = {}
# for b in znames:
#     bs, be = zrepr[b]
#     best, best_span = None, float('inf')
#     for a in znames:
#         if a == b:
#             continue
#         a_s, a_e = zrepr[a]
#         if a_s <= bs and be <= a_e and (a_e - a_s) < best_span:
#             best, best_span = a, a_e - a_s
#     parents[b] = best

# kids = defaultdict(list)
# roots = []
# for zk in znames:
#     if parents[zk] is None:
#         roots.append(zk)
#     else:
#         kids[parents[zk]].append(zk)

# for p in kids:
#     kids[p].sort(key=lambda z: zrepr[z][0])

# # ── 4. Print tree ──
# lines = []

# def _tree(node, depth=0):
#     info = ztotals[node]
#     ms, pct = info['ms'], 100.0 * info['ms'] / total_step_ms
#     cnt = info['count']
#     pre = "- " * (depth + 1)
#     cnt_s = f" (×{cnt})" if cnt > 1 else ""
#     lines.append(f"{pre}{node}{cnt_s}  [{ms:.2f} ms] [{pct:.1f}%]")

#     ch = kids.get(node, [])
#     child_ms = sum(ztotals[c]['ms'] for c in ch)

#     for c in ch:
#         _tree(c, depth + 1)

#     if ch:
#         other = ms - child_ms
#         other_pct = 100.0 * other / total_step_ms if total_step_ms else 0
#         pre2 = "- " * (depth + 2)
#         lines.append(f"{pre2}Other  [{other:.2f} ms] [{other_pct:.1f}%]")

# for r in roots:
#     _tree(r)

# print(f"Device {ZONE_DEVICE_ID}, Iteration {ZONE_ITERATION}\n")
# print("\n".join(lines))